![LogoUC3M](https://upload.wikimedia.org/wikipedia/commons/thumb/a/a6/Acr%C3%B3nimo_y_nombre_de_la_UC3M.svg/320px-Acr%C3%B3nimo_y_nombre_de_la_UC3M.svg.png)

# PRIMERA PRACTICA - ENTRENAMIENTO Y ANALISIS DEL MODELO

### **Grupo 82 - Equipo 15**

*   Ariana Cornejo Infante,     100522121, 100522121@alumnos.uc3m.es
*   Francisco Pérez Sokolowski, 100522254, 100522254@alumnos.uc3m.es

# 2. EDA SIMPLIFICADO

In [ ]:
import pandas as pd
import numpy as np
import requests
import io
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Configuración y Carga de Datos
# Fijamos la semilla (NIA ejemplo: 100522121)
SEED = 100522121
np.random.seed(SEED)

# URLs de los archivos (usando el enlace raw de GitHub)
url_train = "https://github.com/100522254/Proyecto-Aprendizaje-Autom-tico/raw/main/bank_21.pkl"
url_comp = "https://github.com/100522254/Proyecto-Aprendizaje-Autom-tico/raw/main/bank_competition.pkl"

def load_pkl_from_url(url):
    """Carga un archivo .pkl desde una URL y lo devuelve como un DataFrame."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        return pd.read_pickle(io.BytesIO(response.content))
    except Exception as e:
        print(f"Error cargando {url}: {e}")
        return None

print("Cargando datos...")
df = load_pkl_from_url(url_train)
df_competition = load_pkl_from_url(url_comp)

if df is not None:
    print("Datos cargados con éxito.\n")

    # 2. EDA SIMPLIFICADO
    print("--- EDA SIMPLIFICADO ---")

    # Número de Instancias y Variables
    n_instancias, n_variables = df.shape
    print(f"1. Instancias: {n_instancias}")
    print(f"2. Variables: {n_variables}")

    # Tipos de variables
    num_vars = df.select_dtypes(include=['number']).columns.tolist()
    cat_vars = df.select_dtypes(include=['object', 'category']).columns.tolist()
    print(f"3. Variables Numéricas ({len(num_vars)}): {num_vars}")
    print(f"4. Variables Categóricas ({len(cat_vars)}): {cat_vars}")

    # Alta Cardinalidad (>10 unique values)
    high_card = [col for col in cat_vars if df[col].nunique() > 10]
    print(f"5. Categóricas con alta cardinalidad (>10): {high_card}")

    # Valores Faltantes
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"6. Valores faltantes:\n{missing if not missing.empty else 'No hay valores faltantes'}")

    # Columnas constantes o ID
    const_cols = [col for col in df.columns if df[col].nunique() <= 1]
    id_cols = [col for col in df.columns if df[col].nunique() == n_instancias]
    print(f"7. Columnas constantes: {const_cols}")
    print(f"8. Posibles columnas ID: {id_cols}")

    # Problema de Clasificación y Balanceo
    target = 'deposit'
    if target in df.columns:
        print(f"9. Tipo de problema: Clasificación (Variable objetivo: '{target}')")
        balance = df[target].value_counts(normalize=True) * 100
        print(f"10. Balanceo de clases (%):\n{balance}")
        is_imbalanced = any(balance < 20) # Umbral común del 20%
        print(f"    ¿Está desbalanceado? {'Sí' if is_imbalanced else 'No (está relativamente balanceado)'}")

    # 3. ANÁLISIS PARTICULAR DE 'pdays'
    print("\n--- ANÁLISIS DE 'pdays' ---")
    if 'pdays' in df.columns:
        print(df['pdays'].describe())
        minus_one_count = (df['pdays'] == -1).sum()
        print(f"Valores '-1' (nunca contactado): {minus_one_count} ({minus_one_count/n_instancias*100:.2f}%)")

        # Ejemplo de preproceso inmediato:
        df['pdays_contacted'] = np.where(df['pdays'] == -1, 0, 1)
        # Para modelos lineales, a veces es mejor sustituir -1 por un valor que no distorsione (ej. 999 o 0 tras flag)
        # En árboles de decisión, el flag suele ser suficiente.
        print("Preproceso realizado: Se ha añadido la columna 'pdays_contacted'.")

else:
    print("No se pudieron cargar los datos. Verifica los enlaces.")

Cargando datos...
Datos cargados con éxito.

--- EDA SIMPLIFICADO ---
1. Instancias: 11000
2. Variables: 17
3. Variables Numéricas (7): ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
4. Variables Categóricas (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'deposit']
5. Categóricas con alta cardinalidad (>10): ['job', 'month']
6. Valores faltantes:
job          211
education     93
dtype: int64
7. Columnas constantes: []
8. Posibles columnas ID: []
9. Tipo de problema: Clasificación (Variable objetivo: 'deposit')
10. Balanceo de clases (%):
deposit
no     52.545455
yes    47.454545
Name: proportion, dtype: float64
    ¿Está desbalanceado? No (está relativamente balanceado)

--- ANÁLISIS DE 'pdays' ---
count    11000.000000
mean        51.308636
std        108.782842
min         -1.000000
25%         -1.000000
50%         -1.000000
75%         20.250000
max        854.000000
Name: pdays, dtype: float64
Valores '-

## Propuesta de preproceso

**Análisis de *pdays***: La variable 'pdays' representa los días transcurridos desde el último contacto. El valor -1 indica que el cliente nunca fue contactado. Esto no es una relación lineal simple.

**Estrategia:** Crear una variable binaria 'pdays_contacted' (0 si -1, 1 si > 0)
y transformar los -1 en un valor muy alto o usar técnicas de imputación escalado específicas.

In [ ]:
from sklearn.model_selection import train_test_split

# Definimos X (características) y y (lo que queremos predecir)
X = df.drop(columns=['deposit'])
y = df['deposit']

# Dividimos según el enunciado: train (2/3) y test (1/3)
# Usamos stratify=y para mantener ese 52/47 en ambos grupos
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.33,
    random_state=SEED,
    stratify=y
)

print(f"Instancias para entrenar (2/3): {len(X_train)}")
print(f"Instancias para el test final (1/3): {len(X_test)}")

Instancias para entrenar (2/3): 7370
Instancias para el test final (1/3): 3630


# 3. Estrategia de evaluación

Para evaluar el rendimiento del modelo se utilizará una estrategia con dos niveles: outer e inner. La evaluación outer se realizará mediante Holdout, dividiendo el dataset en train (2/3) y test (1/3). El conjunto de entrenamiento se utilizará para desarrollar el modelo, mientras que el conjunto de test se reservará únicamente para la evaluación final, con el objetivo de estimar el rendimiento del modelo en datos no vistos.

Como métrica principal se utilizará F1-Score, ya que combina precisión (precision) y recall en una única métrica mediante su media armónica. Esta métrica es especialmente útil en problemas de clasificación cuando se quiere equilibrar los errores entre falsos positivos y falsos negativos. Para la evaluación interna (inner) se utilizará Stratified 5-Fold Cross Validation, que permitirá comparar diferentes modelos y optimizar hiperparámetros manteniendo la proporción de clases en cada partición.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

target = "deposit"

# Separar variables predictoras y variable objetivo
X = df.drop(columns=[target])
y = df[target]

# Evaluación OUTER (Holdout 2/3 - 1/3)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=1/3,
    stratify=y,
    random_state=SEED
)

print("Tamaño de train:", X_train.shape)
print("Tamaño de test:", X_test.shape)

# Evaluación INNER (Cross Validation)
cv_inner = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)